# Robot Agents with RAI

<p align="center">
  <img src="images/rai_thumb.png" width="900">
</p>

RAI (Robot Agent Interface) is an open-source framework from Robotec.ai for building and deploying Embodied AI agents on robots. It connects a model to a robot's ROS 2 stack, letting it perceive the scene, reason on a natural-language instruction, and call the robot's tools to carry it out.

In this notebook the robot is a Franka Panda arm in an O3DE simulation, and the model behind the agent is the same Lemonade server from the previous notebook. Reasoning therefore continues to run on your Radeon GPU, keeping everything private.

## Goals

* Learn the ROS 2 stack behind a real manipulation pipeline: the camera topics the robot perceives through, the perception services that turn an image into object positions, and MoveIt planning motion around them
* See how an embodied agent is wired onto that stack, with RAI binding ROS 2 tools to a model inside a LangGraph loop, and where the agent's system prompt actually comes from
* Command a simulated Franka Panda arm in natural language, then read the tool-call transcript to see how one sentence became a trajectory
* Run the whole pipeline against a locally served model, so that the reasoning never leaves your machine, keeping your data private

## How the demo fits together

Four pieces run side by side inside this container:

| Piece | Role |
| --- | --- |
| **O3DE simulation** | Physics + rendering of the arm, the table and the objects; publishes camera images and accepts joint commands over ROS 2 |
| **ROS 2 stack** | MoveIt for motion planning, plus the GroundingDINO and SAM 2 perception services the agent uses to find objects |
| **RAI agent** | Turns your instruction into a sequence of tool calls (look at the camera, locate objects, move the arm) |
| **Lemonade** | Serves the model that does the reasoning |

The agent never gets object coordinates for free: it takes a picture through the simulated camera, runs detection on it, and plans from what it sees, the same loop a real robot would run.

## Start the demo

Everything runs inside this notebook: the simulation, the ROS 2 stack, the agent and the chat. The cell below brings all of it up and renders the demo as its own output, so there is no second tab to open and no separate server to reach.

Give it a few minutes the first time. The scene has to render on the iGPU, and MoveIt and the perception services have to come up before the arm can act on anything.

#### (Optional) - Use a different local model

`build_demo` takes a `model=` argument naming the Lemonade model the agent reasons with. It defaults to **Gemma-4-E2B-it-GGUF**, and `layout=` picks the starting scene.


> **Note:** Notice that the detection tool cannot tell colors apart, so the agent has to ask for a category such as "cube" and then pick the red one out of the camera image itself.

In [ ]:
import sys

# Demo helpers live in the workshop scripts directory.
sys.path.insert(0, "/ryzers/notebooks/scripts")

# Importing this initializes rclpy before torch gets loaded - the order matters;
# see the note at the top of scripts/notebook_demo.py.
from notebook_demo import build_demo

demo = build_demo(layout="3 Red Cubes")
demo.display()


## Suggested Prompts

Type into the box under the chat log to command the robot agent; each instruction is reasoned about and executed by your local Gemma model:

- **Reason first:** "What objects can you see on the table, and where are they?"

- **Relocate:** "Move the red cube to the left side of the table."

- **Stack objects:** "Stack any cube on top of another."

- **Sort by color:** "Group the cubes by color."

Watch the tool calls the agent prints as it works: it takes a camera image, calls `get_object_positions` to locate what it saw, and then issues arm movements. When the agent makes an error, such as a missed detection or a grasp the planner couldn't reach, it shows up in the logs in the chat panel.

Use the controls on the left to load a different scene layout or to clear the conversation history before trying another instruction.


## Behind the scenes

![](images/rai_architecture.png)

The system is built from four processes that collaborate in a single loop. O3DE publishes what the camera sees. RAI connects the VLM agent to that stream, feeding it the camera frames so that it can decide what to do next. MoveIt then turns that decision into a trajectory, and the controllers feed joint states back to the simulation. Lemonade sits off to the side as the piece serving the model that does the reasoning. All of the AI and system requests run locally on your machine.

The subsections below take each piece in turn.

### O3DE simulation

O3DE is the simulation that creates the environment for the arm manipulation. It is the one block you could swap out for real hardware, leveraging the same system. It renders the scene on the iGPU and publishes what the camera sees on the topics `/color_image5`, `/depth_image5` and `/color_camera_info5`, along with the arm's joint states. The O3DE node also subscribes to the joint commands coming from MoveIt to follow the inverse kinematics for moving the arm. Everything else, perception and planning and the agent, is the stack a physical Panda arm would run unchanged.

In [ ]:
!ros2 topic list --no-daemon --spin-time 3

### Perception: GroundingDINO and SAM 2

To find an object in the scene, two ROS 2 services are chained together: detection and segmentation. `/detection` runs GroundingDINO, an open-vocabulary detector: hand it an image and a text phrase such as "cube", and it returns boxes around the matching objects. GroundingDINO can identify the objects without a trained class list or any retraining, enabling the agent to name objects in plain language. `/segmentation` then runs SAM 2: given those boxes as prompts, it returns a per-pixel mask for each one. Combining the mask with the depth image and the camera intrinsics gives one 3D point per object, transformed out of the camera frame into `panda_link0`.

That chain is what the agent's `get_object_positions` tool calls, and both models run locally on the same iGPU that serves Gemma.

In [ ]:
!ros2 service list --no-daemon --spin-time 3 | grep -E "detection|segmentation"

### Motion planning: MoveIt

The agent asks for a pose, never for a trajectory. `move_group` closes that gap: it solves the inverse kinematics to turn the target pose into joint angles, searches the joint space for a collision-free path to them, and hands the resulting trajectory to the `ros2_control` controllers. The controllers will then execute it and feed these joint states back to the simulation. The `robotic_manipulation` node is the bridge that exposes all of that data back to the agent as a single tool.

When a grasp fails because the planner could not reach it, this is the piece that reported it.

In [ ]:
!ros2 action list

### The RAI agent

The agent is the only piece in the diagram that reasons; everything above is a tool it can call. It draws itself:

In [ ]:
# The agent draws itself. It is a LangGraph state machine with three nodes:
# the model runs, and depending on whether it asked for a tool the graph either
# routes to `tools` and loops back, or stops.
demo.agent

RAI is a LangChain application: `create_agent` builds a handful of LangChain tools, binds them to the model being served by Lemonade, and wraps them in the LangGraph loop drawn above. The model either answers or emits a tool call, the `tools` node runs it, and the result is appended to the conversation and fed back in.

The tools themselves are generic, since `get_ros2_camera_image` reads whatever topic it is handed. The wiring to this particular robot is therefore configuration rather than code. When the agent is built, RAI remaps each tool onto the simulation's actual topic names, `camera_topic`: `/color_image5`, `depth_topic`: `/depth_image5`, `camera_info_topic`: `/color_camera_info5`, and transforms every detection out of the camera frame `RGBDCamera5` into `panda_link0`, the arm's base. That is what connects the tools the agent calls to the rest of the robotic system.

### System prompt

The agent's system prompt is not written in the code. It is generated from an embodiment file, `examples/embodiments/manipulation_embodiment.json`, which describes the robot in four parts: what it is, the rules it must obey, what it is physically capable of, and how it should behave. RAI renders those into the block below and prepends it to every conversation, so this is verbatim the context your Gemma model reads before it ever sees your instruction.

In [ ]:
from rai_whoami.models import EmbodimentInfo

EMBODIMENT = "/ryzers/rai/examples/embodiments/manipulation_embodiment.json"

# to_langchain() builds the SystemMessage the agent prepends to every conversation.
# Its content is a list of multimodal parts, so printing it directly shows the
# dicts and escaped newlines; join the text parts instead to read it as prose.
system_message = EmbodimentInfo.from_file(EMBODIMENT).to_langchain()

print("".join(p["text"] for p in system_message.content if p["type"] == "text"))

> **Note:** Notice that cubes are declared to be 5 cm tall. The perception pipeline returns a single point for each object and reports its size as unknown, so that one line in the prompt is what lets the agent stack cubes at the right height.

### Peek at what the robot sees

The agent's view of the world is the `/color_image5` camera topic. `web_video_server` serves it over HTTP, which is what the simulation panel displays, so we can grab a frame straight from this notebook.

In [ ]:
import requests
import matplotlib.pyplot as plt
from PIL import Image
from io import BytesIO

# web_video_server wants the topic unescaped, so keep it in the URL itself
SNAPSHOT_URL = "http://localhost:8080/snapshot?topic=/color_image5&quality=80"

r = requests.get(SNAPSHOT_URL, timeout=30)
r.raise_for_status()

plt.figure(figsize=(8, 4.5))
plt.imshow(Image.open(BytesIO(r.content)))
plt.axis("off")
plt.title("Live view from the simulated camera")
plt.show()

## Test OpenAI API Server Directly

The chat panel above is only a front end. The agent behind it is a plain object you can build and call yourself, which is worth doing once because it makes the whole loop concrete.

In the previous notebook we POSTed a message to Lemonade and got text back. Here we hand an instruction to the agent, and it makes that same call under the hood, except that the answer arrives as motion: the model decides which tools to call, and the arm moves. We therefore record the simulation camera while it works and play the result back as a video.

Keep the demo from the previous section running, since this uses the simulation and the ROS 2 stack it started.

In [ ]:
import os
import sys

# rclpy.init() must happen before torch is loaded: torch's bundled C++
# runtime corrupts the heap for rcl's init, aborting the kernel with
# "free(): invalid size". RAI's connector skips its own init if rclpy is
# already up, and recommends initializing manually anyway.
import rclpy

if not rclpy.ok():
    rclpy.init()

# RAI resolves its embodiment description and config.toml relative to the repo root
sys.path.insert(0, "/ryzers/rai/examples")
os.chdir("/ryzers/rai")
os.environ.setdefault("OPENAI_API_KEY", "lemonade")  # dummy key, Lemonade ignores it

from manipulation_common import create_agent

# Builds the LLM client (configured by scripts/lemonade_env.sh) and the robot's
# tools: look through the camera, locate objects, move the arm, reset the arm
agent, camera_tool = create_agent(version="v2")

print("Agent ready")

### Send an instruction and record the result

`agent.invoke` blocks until the agent is done, which usually takes four to five minutes, since every step is a full round trip through the local model. Most of that time is the model thinking, so the cell prints a running log of what it is doing, a line per model turn and per tool call, plus a heartbeat from the recorder every 30 seconds. As long as those keep appearing, the run is alive and not wedged.

While the agent works, a background thread pulls frames from the same MJPEG endpoint the simulation panel uses, and encodes them into an MP4 once the run finishes.

Change `INSTRUCTION` to whatever you want the arm to do; keep it to objects that are actually in the current scene layout.

In [ ]:
import subprocess
import threading
import time

import requests  # also imported by the snapshot cell above, which may not have run

from IPython.display import Video, display
from langchain_core.callbacks.base import BaseCallbackHandler
from rai.messages import HumanMultimodalMessage

RECORD_URL = (
    "http://localhost:8080/stream?topic=/color_image5&quality=70&width=960&height=540"
)
VIDEO_PATH = "/ryzers/notebooks/agent_run.mp4"


class ProgressLog(BaseCallbackHandler):
    """Prints what the agent is doing, so a long run is visibly alive.

    A run is minutes of near-silence otherwise: one model turn can take tens of
    seconds, and nothing reaches the notebook until the whole invoke returns.
    """

    def __init__(self):
        self._started = {}
        self._turn = 0
        self._t0 = time.time()

    def _stamp(self):
        return f"[{time.time() - self._t0:5.1f}s]"

    # Chat models fire on_chat_model_start; plain LLMs fire on_llm_start
    def on_chat_model_start(self, serialized, messages, **kwargs):
        self._turn += 1
        print(f"{self._stamp()} thinking (model turn {self._turn})...", flush=True)

    on_llm_start = on_chat_model_start

    def on_tool_start(self, serialized, input_str, *, run_id=None, **kwargs):
        name = (serialized or {}).get("name", "tool")
        self._started[run_id] = (name, time.time())
        print(f"{self._stamp()}   -> {name}({input_str})", flush=True)

    def on_tool_end(self, output, *, run_id=None, **kwargs):
        name, started = self._started.pop(run_id, ("tool", time.time()))
        text = str(output).replace("\n", " ")
        if len(text) > 120:
            text = text[:120] + "..."
        print(f"{self._stamp()}   <- {name} [{time.time() - started:.1f}s] {text}", flush=True)

    def on_tool_error(self, error, *, run_id=None, **kwargs):
        name, _ = self._started.pop(run_id, ("tool", 0))
        print(f"{self._stamp()}   !! {name} failed: {error}", flush=True)


class SimulationRecorder:
    """Capture the simulation camera into an MP4 while the agent works."""

    def __init__(self, path=VIDEO_PATH, fps=4):
        self.path, self.fps = path, fps
        self.frames, self._stop = [], threading.Event()

    def _grab(self):
        # One long-lived MJPEG connection, split into frames on the JPEG markers.
        # (Polling /snapshot per frame instead would churn through subscribers
        # and eventually hang web_video_server.)
        try:
            with requests.get(RECORD_URL, stream=True, timeout=30) as r:
                buf, last = b"", 0.0
                for chunk in r.iter_content(8192):
                    if self._stop.is_set():
                        return
                    buf += chunk
                    while True:
                        start, end = buf.find(b"\xff\xd8"), buf.find(b"\xff\xd9")
                        if start == -1 or end == -1 or end < start:
                            break
                        frame, buf = buf[start : end + 2], buf[end + 2 :]
                        now = time.time()
                        if now - last >= 1 / self.fps:  # thin out to the target rate
                            self.frames.append(frame)
                            last = now
        except requests.RequestException:
            pass

    def _heartbeat(self):
        # Own thread rather than a check inside _grab: the point is to keep
        # reporting even when web_video_server stops delivering frames.
        while not self._stop.wait(30):
            print(
                f"          ... {time.time() - self._started:.0f}s elapsed, "
                f"{len(self.frames)} frames captured, agent still working",
                flush=True,
            )

    def __enter__(self):
        self._started = time.time()
        self._thread = threading.Thread(target=self._grab, daemon=True)
        self._thread.start()
        self._pulse = threading.Thread(target=self._heartbeat, daemon=True)
        self._pulse.start()
        print("Recording the simulation. This usually takes four to five minutes.", flush=True)
        return self

    def __exit__(self, *exc):
        self._stop.set()
        self._thread.join(timeout=5)
        self._pulse.join(timeout=1)
        if not self.frames:
            # The grab thread blocks in iter_content when web_video_server stops
            # answering, so joining it times out and no frames ever arrive.
            print(
                "No frames captured - web_video_server is not responding.\n"
                "Restart it from a terminal:\n"
                "  pkill -9 -f web_video_server\n"
                "  source /opt/ros/$ROS_DISTRO/setup.bash\n"
                "  setsid ros2 run web_video_server web_video_server "
                "--ros-args -p port:=8080 </dev/null >/tmp/wvs.log 2>&1 &"
            )
            return
        # Encode at the rate we actually captured, so playback runs at real speed
        print(f"Agent finished. Encoding {len(self.frames)} frames...", flush=True)
        rate = max(len(self.frames) / (time.time() - self._started), 1)
        ffmpeg = subprocess.Popen(
            ["ffmpeg", "-y", "-loglevel", "error",
             "-f", "image2pipe", "-vcodec", "mjpeg", "-framerate", f"{rate:.2f}", "-i", "-",
             "-vcodec", "libx264", "-pix_fmt", "yuv420p", "-vf", "fps=15", self.path],
            stdin=subprocess.PIPE,
        )
        ffmpeg.communicate(b"".join(self.frames))
        print(f"Video ready: {self.path}", flush=True)

In [ ]:
INSTRUCTION = "Pick up any cube and place it on top of another cube."

# The agent starts from what the camera sees, exactly like the chat panel does
_, artifact = camera_tool._run()
message = HumanMultimodalMessage(content=INSTRUCTION, images=artifact.get("images", []))

start = time.time()
with SimulationRecorder():
    result = agent.invoke(
        {"messages": [message]},
        config={"recursion_limit": 100, "callbacks": [ProgressLog()]},
    )

print(f"\n{result['messages'][-1].content}\n\n(took {time.time() - start:.0f}s)")
display(Video(VIDEO_PATH, embed=True, width=640))

#### Optional: Agent Tools Called

The full transcript of the run is kept in `result["messages"]`: your instruction, every tool the agent decided to call, the arguments it passed, and what each tool returned. Printing it shows exactly how the agent got from a sentence to the motion you just watched.

In [ ]:
for m in result["messages"]:
    m.pretty_print()

## Key Takeaways

Now you know:
- How a robot agent is wired: simulation and ROS 2 tools on one side, a locally served LLM on the other
- What the manipulation pipeline is made of: GroundingDINO and SAM 2 turning a phrase into a 3D point, and MoveIt turning a pose into a trajectory
- How to point RAI at a Lemonade model with `scripts/lemonade_env.sh`, and swap that model out
- How to run a GPU-rendered simulation with no monitor, using a headless compositor plus Xwayland
- How to command a simulated manipulator in natural language, and read the agent's tool calls to see how it got there

## What to Try Next

- Ask for a task the arm can't reach or see, and watch how the agent recovers
- Swap in a different local model and compare how reliably each one calls the right tools
- Change the scene layout in the controls on the left and repeat the same instruction
- Explore RAI's benchmarks (`rai_bench`) to score models on tool calling and manipulation with your local model

## References

* [RAI](https://github.com/RobotecAI/rai)
* [O3DE](https://o3de.org/)
* [Lemonade](https://lemonade-server.ai/)

**Continue to**: [3_code_as_policy.ipynb](3_code_as_policy.ipynb)

---
Copyright© 2026 AMD, Inc SPDX-License-Identifier: MIT